In [3]:
import pandas as pd
import numpy as np


DATE_COLS = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate"
]

NUMERIC_COLS = [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "Latitude",
    "Longitude"
]

#
UNNECESSARY_COLS = [
    "Unnamed: 0",
    "SourceSystemID",
    "OriginatingSystemName",
    "OriginatingSystemKey",
    "ModificationTimestamp",
    "PhotosChangeTimestamp"
]


def clean_mls_data(file_path, output_path):
    df = pd.read_csv(file_path, low_memory=False)

    # -----------------------------
    # 1. Convert date fields
    # -----------------------------
    for col in DATE_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # -----------------------------
    # 2. Remove unnecessary columns
    # -----------------------------
    cols_to_drop = [col for col in UNNECESSARY_COLS if col in df.columns]
    df = df.drop(columns=cols_to_drop)

    # -----------------------------
    # 3. Ensure numeric fields are typed correctly
    # -----------------------------
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # -----------------------------
    # 4. Invalid numeric value flags
    # -----------------------------
    if "ClosePrice" in df.columns:
        df["invalid_close_price_flag"] = df["ClosePrice"] <= 0
    else:
        df["invalid_close_price_flag"] = False

    if "LivingArea" in df.columns:
        df["invalid_living_area_flag"] = df["LivingArea"] <= 0
    else:
        df["invalid_living_area_flag"] = False

    if "DaysOnMarket" in df.columns:
        df["invalid_days_on_market_flag"] = df["DaysOnMarket"] < 0
    else:
        df["invalid_days_on_market_flag"] = False

    if "BedroomsTotal" in df.columns:
        df["invalid_bedrooms_flag"] = df["BedroomsTotal"] < 0
    else:
        df["invalid_bedrooms_flag"] = False

    if "BathroomsTotalInteger" in df.columns:
        df["invalid_bathrooms_flag"] = df["BathroomsTotalInteger"] < 0
    else:
        df["invalid_bathrooms_flag"] = False

    # -----------------------------
    # 5. Date consistency checks
    # -----------------------------

    # ListingContractDate should be before CloseDate
    if {"ListingContractDate", "CloseDate"}.issubset(df.columns):
        df["listing_after_close_flag"] = (
            df["ListingContractDate"].notna()
            & df["CloseDate"].notna()
            & (df["ListingContractDate"] > df["CloseDate"])
        )
    else:
        df["listing_after_close_flag"] = False

    # PurchaseContractDate should be before CloseDate
    if {"PurchaseContractDate", "CloseDate"}.issubset(df.columns):
        df["purchase_after_close_flag"] = (
            df["PurchaseContractDate"].notna()
            & df["CloseDate"].notna()
            & (df["PurchaseContractDate"] > df["CloseDate"])
        )
    else:
        df["purchase_after_close_flag"] = False

    # ListingContractDate should be before PurchaseContractDate
    if {"ListingContractDate", "PurchaseContractDate"}.issubset(df.columns):
        df["negative_timeline_flag"] = (
            df["ListingContractDate"].notna()
            & df["PurchaseContractDate"].notna()
            & (df["ListingContractDate"] > df["PurchaseContractDate"])
        )
    else:
        df["negative_timeline_flag"] = False

    # -----------------------------
    # 6. Geographic data checks
    # -----------------------------

    if {"Latitude", "Longitude"}.issubset(df.columns):

        df["missing_coordinates_flag"] = (
            df["Latitude"].isna() | df["Longitude"].isna()
        )

        df["zero_coordinates_flag"] = (
            (df["Latitude"] == 0) | (df["Longitude"] == 0)
        )

        # California longitude should be negative
        df["positive_longitude_flag"] = df["Longitude"] > 0

        # Approximate California coordinate range
        # Latitude: about 32 to 42
        # Longitude: about -125 to -114
        df["implausible_coordinates_flag"] = (
            df["Latitude"].notna()
            & df["Longitude"].notna()
            & (
                (df["Latitude"] < 32)
                | (df["Latitude"] > 42)
                | (df["Longitude"] < -125)
                | (df["Longitude"] > -114)
            )
        )

    else:
        df["missing_coordinates_flag"] = False
        df["zero_coordinates_flag"] = False
        df["positive_longitude_flag"] = False
        df["implausible_coordinates_flag"] = False

    # -----------------------------
    # 7. Overall invalid row flag
    # -----------------------------
    flag_cols = [col for col in df.columns if col.endswith("_flag")]

    df["any_data_quality_issue_flag"] = df[flag_cols].any(axis=1)

    # -----------------------------
    # 8. Handle missing values
    # -----------------------------
    # For now, this keeps missing values as NaN.
    # This is usually safer before modeling because imputation depends on the analysis goal.
    #
    # Example optional imputations:
    # df["LivingArea"] = df["LivingArea"].fillna(df["LivingArea"].median())
    # df["BedroomsTotal"] = df["BedroomsTotal"].fillna(0)

    # -----------------------------
    # 9. Save cleaned data
    # -----------------------------
    df.to_csv(output_path, index=False)

    print(f"Finished cleaning: {file_path}")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    print(f"Saved to: {output_path}")
    print()
    print("Data quality issue counts:")
    print(df[flag_cols].sum().sort_values(ascending=False))

    return df


listing_clean = clean_mls_data(
    "data/listing.csv",
    "listing_cleaned.csv"
)

sold_clean = clean_mls_data(
    "data/sold.csv",
    "sold_cleaned.csv"
)

Finished cleaning: data/listing.csv
Rows: 540183
Columns: 97
Saved to: listing_cleaned.csv

Data quality issue counts:
missing_coordinates_flag        80145
invalid_living_area_flag          359
implausible_coordinates_flag      289
negative_timeline_flag            271
purchase_after_close_flag         265
positive_longitude_flag            85
listing_after_close_flag           72
zero_coordinates_flag              60
invalid_days_on_market_flag        29
invalid_close_price_flag            0
invalid_bedrooms_flag               0
invalid_bathrooms_flag              0
dtype: int64
Finished cleaning: data/sold.csv
Rows: 397603
Columns: 96
Saved to: sold_cleaned.csv

Data quality issue counts:
missing_coordinates_flag        15822
negative_timeline_flag            261
purchase_after_close_flag         240
invalid_living_area_flag          144
implausible_coordinates_flag       84
listing_after_close_flag           58
invalid_days_on_market_flag        46
positive_longitude_flag          